# 01 Momentum Research — Macro Metals System

> **Strategy:** Canonical Time-Series Momentum (TSMOM)
> **Reference:** Moskowitz, Ooi & Pedersen (2012); Quantpedia; JPM/CME report
> **Scope:** In-sample development (2015–2022), monthly rebalance
> **Instruments:** Gold, Silver, Platinum futures · EURUSD, USDJPY, AUDUSD · SOFR front quartet
> **Signal:** sign(12-month return), monthly rebalance, inverse-vol position sizing
>
> Self-contained BQuant notebook — executable top-to-bottom.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yaml
from pathlib import Path
from datetime import datetime

# Bloomberg BQL
import bql
bq = bql.Service()

print(f"Session started : {datetime.now():%Y-%m-%d %H:%M}")
print(f"BQL service     : {type(bq).__name__}")
print(f"NumPy {np.__version__}  |  pandas {pd.__version__}")

## Config & Parameters

Load `parameters.yaml` and `tickers.yaml` from `config/`.

**Canonical TSMOM parameters** (defaults if missing from YAML):

| Parameter | Default | Description |
|-----------|---------|-------------|
| `lookback_days` | 252 | 12-month return lookback |
| `rebalance_freq` | monthly | Signal update frequency |
| `target_vol_annual` | 0.10 | Annualised vol target per instrument |
| `vol_lookback_days` | 30 | Not used — EWMA λ controls vol estimation |
| `vol_decay_lambda` | 0.94 | EWMA decay for variance estimation |
| `leverage_cap_multiplier` | 2.0 | Max position scaling factor |
| `transaction_cost_bp_per_side` | 2.0 | One-way cost in basis points |

In [ ]:
CONFIG_DIR = Path("config")

with open(CONFIG_DIR / "parameters.yaml") as f:
    params = yaml.safe_load(f)
with open(CONFIG_DIR / "tickers.yaml") as f:
    tickers = yaml.safe_load(f)

gcfg    = params["global"]
mom_cfg = params["strategies"]["momentum"]
targets = params["performance_targets"]

# ── Canonical TSMOM parameters (with safe defaults) ──────────────────
def _get(cfg, key, default, label=""):
    if key in cfg:
        return cfg[key]
    print(f"  ⚠ {label or key} missing from config — using default {default}")
    return default

LOOKBACK_DAYS   = 252                                              # always 12M
REBALANCE_FREQ  = "monthly"                                        # always monthly
TARGET_VOL      = _get(gcfg, "target_portfolio_vol_annual", 0.10, "target_vol_annual")
VOL_LAMBDA      = _get(gcfg, "vol_decay_lambda", 0.94)
LEV_CAP         = _get(gcfg, "vol_cap_multiplier", 2.0, "leverage_cap_multiplier")
TC_BP           = 2.0                                              # per side

# Flatten instrument universe from config groups
UNIVERSE = {}
for asset_class, keys in mom_cfg["instruments"].items():
    for k in keys:
        UNIVERSE[k] = asset_class

LABELS = {
    "gc_fut_front":    "Gold (GC)",
    "si_fut_front":    "Silver (SI)",
    "pl_fut_front":    "Platinum (PL)",
    "eurusd_spot":     "EURUSD",
    "usdjpy_spot":     "USDJPY",
    "audusd_spot":     "AUDUSD",
    "sofr_fut_front":  "SOFR Front",
    "sofr_fut_second": "SOFR 2nd",
    "sofr_fut_third":  "SOFR 3rd",
    "sofr_fut_fourth": "SOFR 4th",
}

IS_START = gcfg["in_sample_start"]
IS_END   = gcfg["in_sample_end"]

print("\nCanonical TSMOM parameters:")
print(f"  Lookback       : {LOOKBACK_DAYS}d (12 months)")
print(f"  Rebalance      : {REBALANCE_FREQ}")
print(f"  Vol target     : {TARGET_VOL:.0%}")
print(f"  EWMA λ         : {VOL_LAMBDA}")
print(f"  Leverage cap   : {LEV_CAP:.1f}x")
print(f"  TC per side    : {TC_BP:.1f} bp")
print(f"  IS period      : {IS_START} to {IS_END}")
print(f"  Universe       : {len(UNIVERSE)} instruments")

## Data Pipeline (BQL)

Fetch daily `PX_LAST` for the full momentum universe via Bloomberg BQL.
Uses the **corrected BQL syntax** — `df.set_index('DATE')` (BQL returns
a flat DataFrame with a `DATE` column, not a MultiIndex).

In [ ]:
class BQuantDataLoader:
    """Fetch historical prices via Bloomberg BQL.

    Uses corrected BQL response handling: df.set_index('DATE').
    """

    def __init__(self, ticker_map: dict) -> None:
        self._tickers = ticker_map
        self._bq = bql.Service()

    def resolve(self, logical_name: str) -> str:
        """Logical name -> Bloomberg ticker string."""
        for group in self._tickers.values():
            if isinstance(group, dict) and logical_name in group:
                return group[logical_name]
        raise KeyError(f"{logical_name} not in tickers.yaml")

    def get_history(
        self,
        logical_name: str,
        start: str,
        end: str,
        field: str = "PX_LAST",
    ) -> pd.Series:
        """Fetch daily price series for one instrument."""
        bbg = self.resolve(logical_name)
        request = bql.Request(
            bbg,
            {field: self._bq.data.px_last(
                dates=self._bq.func.range(start, end)
            )},
        )
        try:
            response = self._bq.execute(request)
            df = response[0].df()
            if df.empty:
                print(f"  ! {logical_name}: empty response")
                return pd.Series(dtype=float, name=logical_name)

            # BQL returns flat DataFrame with DATE column
            df_fixed = df.set_index('DATE')
            series = df_fixed[field]

            # Convert index to datetime, handle invalid dates
            series.index = pd.to_datetime(series.index, errors='coerce')
            series = series.dropna()

            # Remove duplicate dates
            series = series[~series.index.duplicated(keep='last')]

            series = series.sort_index().astype(float)
            series.name = logical_name
            series.index.name = "date"

            if not isinstance(series.index, pd.DatetimeIndex):
                series.index = pd.to_datetime(series.index)

            return series
        except Exception as exc:
            print(f"  ! {logical_name} ({bbg}): {exc}")
            return pd.Series(dtype=float, name=logical_name)


# --- Fetch all instruments ---
loader = BQuantDataLoader(tickers)
prices = {}

for key in UNIVERSE:
    label = LABELS.get(key, key)
    print(f"  {label:20s}", end=" ")
    s = loader.get_history(key, IS_START, IS_END)
    if len(s) > 0:
        prices[key] = s
        print(f"OK  {len(s):>5d} obs  [{s.index[0]} -> {s.index[-1]}]")
    else:
        print("MISSING")

# Clean: deduplicate indices
cleaned_prices = {}
for key, series in prices.items():
    cleaned = series[~series.index.duplicated(keep='last')]
    cleaned_prices[key] = cleaned
    if len(series) != len(cleaned):
        print(f"  {key}: removed {len(series) - len(cleaned)} duplicate dates")

# Build aligned panel
prices_df = pd.DataFrame(cleaned_prices).sort_index()
prices_df = prices_df[prices_df.index.notna()]
prices_df = prices_df.ffill()

# Drop instruments with insufficient history for 252d lookback
min_obs = LOOKBACK_DAYS + 30  # need 252 + buffer
sufficient = prices_df.count() >= min_obs
dropped = prices_df.columns[~sufficient].tolist()
if dropped:
    print(f"\n  Dropping (insufficient history): {dropped}")
    prices_df = prices_df[prices_df.columns[sufficient]]

print(f"\nPanel: {prices_df.shape[0]} days x {prices_df.shape[1]} instruments")
prices_df.tail(3)

## Canonical TSMOM Strategy

**Moskowitz, Ooi & Pedersen (2012)** implementation:

1. **Signal:** `sign(12-month return)` — go long if past 12M return > 0, short if < 0
2. **Rebalance:** Monthly (month-end signal applied on first trading day of next month)
3. **Position sizing:** Inverse-volatility scaling to target 10% annualised vol per instrument
4. **Volatility estimate:** EWMA with λ = 0.94 (RiskMetrics standard)
5. **Leverage cap:** 2.0× per instrument

No discretionary overlays: no dead zone, no tanh normalisation, no EMA crossover,
no Donchian breakout. Pure sign-of-return signal.

In [ ]:
class CanonicalTSMOMStrategy:
    """Canonical Time-Series Momentum (Moskowitz, Ooi & Pedersen 2012).

    Signal = sign(12-month return), monthly rebalance,
    inverse-vol position sizing.
    """

    def __init__(
        self,
        lookback_days: int = 252,
        target_vol: float = 0.10,
        vol_lambda: float = 0.94,
        leverage_cap: float = 2.0,
    ) -> None:
        self.lookback_days = lookback_days
        self.target_vol    = target_vol
        self.vol_lambda    = vol_lambda
        self.leverage_cap  = leverage_cap

    # ── Volatility estimation ──────────────────────────────────────

    def ex_ante_vol_ewma(self, returns: pd.DataFrame) -> pd.DataFrame:
        """EWMA volatility estimate (RiskMetrics style).

        Uses alpha = 1 - lambda on squared returns, then sqrt.
        Returns annualised volatility.
        """
        alpha = 1 - self.vol_lambda
        # EWMA of squared returns → daily variance
        ewma_var = returns.pow(2).ewm(alpha=alpha, adjust=False).mean()
        vol_daily = np.sqrt(ewma_var)
        vol_ann = vol_daily * np.sqrt(252)
        return vol_ann

    # ── Signal generation ──────────────────────────────────────────

    def compute_monthly_signal(self, prices_df: pd.DataFrame) -> pd.DataFrame:
        """Compute canonical TSMOM monthly signal.

        Steps:
            1. 12-month return → sign → daily signal grid
            2. Resample to month-end, shift by 1 month (no lookahead)
            3. Forward-fill within each month → constant intra-month signal

        Returns:
            DataFrame of daily signals in {-1, 0, +1}, same shape as prices_df.
        """
        # 1. Daily 12M return and its sign
        r12 = prices_df.pct_change(self.lookback_days)
        sig_daily = np.sign(r12)

        # Avoid lookahead: shift by 1 day
        sig_daily = sig_daily.shift(1)

        # 2. Month-end signal snapshot
        month_end_signal = sig_daily.resample("M").last()

        # Shift by 1 month: signal computed at month-end
        # is applied starting the first trading day of the NEXT month
        trade_signal = month_end_signal.shift(1)

        # 3. Forward-fill to daily frequency
        daily_signal = trade_signal.reindex(prices_df.index).ffill().fillna(0.0)

        return daily_signal

    # ── Position sizing ────────────────────────────────────────────

    def compute_positions(
        self,
        prices_df: pd.DataFrame,
        signals: pd.DataFrame,
    ) -> pd.DataFrame:
        """Vol-scaled positions with monthly rebalance.

        Position = signal × (target_vol / ex-ante_vol), capped at ±leverage_cap.
        Vol sampled at month-end and held constant intra-month.
        """
        ret = prices_df.pct_change().fillna(0.0)
        vol_ann = self.ex_ante_vol_ewma(ret)

        # Sample vol at month-end
        vol_month_end = vol_ann.resample("M").last()

        # Shift to align with next month's positions
        vol_for_sizing = vol_month_end.shift(1)

        # Forward-fill to daily
        vol_daily = vol_for_sizing.reindex(prices_df.index).ffill()

        # Replace zero/NaN vol to avoid division errors
        vol_daily = vol_daily.replace(0, np.nan)

        # Position = signal × (target / vol), capped
        raw_position = signals * (self.target_vol / vol_daily)
        position = raw_position.clip(
            -self.leverage_cap, self.leverage_cap
        ).fillna(0.0)

        return position


# Build strategy
tsmom = CanonicalTSMOMStrategy(
    lookback_days=LOOKBACK_DAYS,
    target_vol=TARGET_VOL,
    vol_lambda=VOL_LAMBDA,
    leverage_cap=LEV_CAP,
)
print("CanonicalTSMOMStrategy initialised.")
print(f"  Lookback:  {tsmom.lookback_days}d")
print(f"  Target:    {tsmom.target_vol:.0%} annual vol")
print(f"  Lambda:    {tsmom.vol_lambda}")
print(f"  Lev cap:   {tsmom.leverage_cap}x")

## Generate Signals & Positions (IS Period)

Monthly signal from sign(12M return) → vol-scaled positions.

In [ ]:
# Generate signals and positions
signals = tsmom.compute_monthly_signal(prices_df)
positions = tsmom.compute_positions(prices_df, signals)

print(f"Signal matrix : {signals.shape[0]} days x {signals.shape[1]} instruments")
print(f"Position matrix: {positions.shape[0]} days x {positions.shape[1]} instruments")
print(f"Date range    : {signals.index[0]:%Y-%m-%d} to {signals.index[-1]:%Y-%m-%d}")

# ── Sanity check 1: Signal distribution ──────────────────────────────
print("\n" + "=" * 65)
print("  SIGNAL DISTRIBUTION (% of days)")
print("=" * 65)
for col in signals.columns:
    s = signals[col]
    n = len(s)
    pct_long  = (s ==  1).sum() / n * 100
    pct_short = (s == -1).sum() / n * 100
    pct_flat  = (s ==  0).sum() / n * 100
    label = LABELS.get(col, col)
    print(f"  {label:20s}  Long={pct_long:5.1f}%  Short={pct_short:5.1f}%  Flat={pct_flat:5.1f}%")

# ── Sanity check 2: Average holding period ───────────────────────────
print("\n" + "=" * 65)
print("  AVERAGE HOLDING PERIOD (trading days)")
print("=" * 65)
for col in signals.columns:
    changes = (signals[col].diff().abs() > 0).sum()
    if changes > 0:
        avg_hold = len(signals[col]) / changes
    else:
        avg_hold = len(signals[col])
    label = LABELS.get(col, col)
    print(f"  {label:20s}  {avg_hold:.0f} days  ({avg_hold/21:.1f} months)")

# ── Sanity check 3: Signals strictly in {-1, 0, +1} ─────────────────
unique_vals = set()
for col in signals.columns:
    unique_vals.update(signals[col].dropna().unique())
print(f"\nUnique signal values: {sorted(unique_vals)}")
assert unique_vals <= {-1.0, 0.0, 1.0}, "Signal values outside {-1, 0, +1}!"
print("✓ Signals are strictly in {-1, 0, +1}")

## Backtest Engine

Monthly-turnover backtest:
- Positions change only at month boundaries (from vol-scaled signal)
- Gross return: `position(t-1) × return(t)`, averaged across instruments
- Transaction costs: 2 bp per side on absolute position changes
- Turnover should show monthly spikes, not daily churn

In [ ]:
def backtest_tsmom(
    prices_df: pd.DataFrame,
    positions: pd.DataFrame,
    tc_bp: float = 2.0,
    capital: float = 1_000_000.0,
) -> dict:
    """Run canonical TSMOM backtest.

    Args:
        prices_df: Daily price panel.
        positions: Daily position panel (changes only monthly).
        tc_bp:     Transaction cost per side in basis points.
        capital:   Starting capital.

    Returns:
        Dict with keys: per_instrument (dict of DataFrames),
        portfolio (DataFrame), metrics_per_inst, metrics_portfolio.
    """
    ret = prices_df.pct_change().fillna(0.0)

    # ── Per-instrument results ────────────────────────────────────
    inst_results = {}
    inst_metrics = []

    for col in positions.columns:
        pos = positions[col]
        r   = ret[col]

        # Gross return: yesterday's position × today's return
        ret_gross = pos.shift(1).fillna(0.0) * r

        # Transaction costs on position changes
        turnover = pos.diff().abs().fillna(0.0)
        cost = turnover * (tc_bp / 10_000)
        ret_net = ret_gross - cost

        equity = capital * (1 + ret_net).cumprod()

        df = pd.DataFrame({
            "position": pos,
            "price": prices_df[col],
            "ret_gross": ret_gross,
            "ret_net": ret_net,
            "equity": equity,
            "turnover": turnover,
        })
        inst_results[col] = df

        # Metrics
        m = _compute_metrics(ret_net, equity, turnover)
        m["Instrument"] = LABELS.get(col, col)
        inst_metrics.append(m)

    # ── Portfolio (equal-weight across instruments) ────────────────
    n_active = positions.ne(0).sum(axis=1).clip(lower=1)

    # Per-instrument gross and net returns
    gross_all = pd.DataFrame()
    net_all   = pd.DataFrame()
    turn_all  = pd.DataFrame()
    for col in positions.columns:
        gross_all[col] = inst_results[col]["ret_gross"]
        net_all[col]   = inst_results[col]["ret_net"]
        turn_all[col]  = inst_results[col]["turnover"]

    # Equal-weight portfolio return
    port_ret_gross = gross_all.sum(axis=1) / positions.shape[1]
    port_ret_net   = net_all.sum(axis=1) / positions.shape[1]
    port_turnover  = turn_all.sum(axis=1)
    port_equity    = capital * (1 + port_ret_net).cumprod()

    port_df = pd.DataFrame({
        "ret_gross": port_ret_gross,
        "ret_net": port_ret_net,
        "equity": port_equity,
        "turnover": port_turnover,
    })

    port_m = _compute_metrics(port_ret_net, port_equity, port_turnover)
    port_m["Instrument"] = "PORTFOLIO"

    return {
        "per_instrument": inst_results,
        "portfolio": port_df,
        "metrics_per_inst": inst_metrics,
        "metrics_portfolio": port_m,
    }


def _compute_metrics(
    ret_net: pd.Series,
    equity: pd.Series,
    turnover: pd.Series,
    periods: int = 252,
) -> dict:
    """Performance metrics (§6.3 targets)."""
    r = ret_net.dropna()
    eq = equity.dropna()

    n_years = len(r) / periods
    total   = (1 + r).prod()
    ann_ret = total ** (1 / max(n_years, 0.01)) - 1
    ann_vol = r.std() * np.sqrt(periods)
    sharpe  = ann_ret / ann_vol if ann_vol > 0 else 0.0

    running_max = eq.cummax()
    dd = (eq - running_max) / running_max
    max_dd = float(-dd.min()) if len(dd) > 0 else 0.0

    calmar = ann_ret / max_dd if max_dd > 0 else 0.0
    hit    = float((r > 0).sum() / len(r)) if len(r) > 0 else 0.0
    ann_to = turnover.sum() / max(n_years, 0.01)

    return {
        "Ann. Return":   ann_ret,
        "Ann. Vol":      ann_vol,
        "Sharpe":        sharpe,
        "Max DD":        max_dd,
        "Calmar":        calmar,
        "Hit Rate":      hit,
        "Ann. Turnover": ann_to,
    }


print("Backtest engine ready (monthly-turnover variant).")

## Results

Run backtest, compute per-instrument and portfolio metrics.

In [ ]:
bt = backtest_tsmom(prices_df, positions, tc_bp=TC_BP)

# Per-instrument metrics table
metrics_df = pd.DataFrame(bt["metrics_per_inst"]).set_index("Instrument")

# Add portfolio row
port_row = pd.DataFrame([bt["metrics_portfolio"]]).set_index("Instrument")
metrics_all = pd.concat([metrics_df, port_row])

# Formatted display
fmt_df = metrics_all.copy()
for col in ["Ann. Return", "Ann. Vol", "Max DD", "Hit Rate"]:
    fmt_df[col] = fmt_df[col].map("{:.1%}".format)
fmt_df["Sharpe"]        = fmt_df["Sharpe"].map("{:.2f}".format)
fmt_df["Calmar"]        = fmt_df["Calmar"].map("{:.2f}".format)
fmt_df["Ann. Turnover"] = fmt_df["Ann. Turnover"].map("{:.1f}x".format)

print("=" * 70)
print("  CANONICAL TSMOM — PER-INSTRUMENT + PORTFOLIO METRICS (IS 2015-2022)")
print("=" * 70)
fmt_df

## Diagnostics

Instrument coverage, volatility, and correlation structure.

In [ ]:
# ── Diagnostic 1: Data coverage ────────────────────────────────────
print("=" * 65)
print("  DATA COVERAGE")
print("=" * 65)
for col in prices_df.columns:
    n_obs = prices_df[col].dropna().shape[0]
    n_missing = prices_df[col].isna().sum()
    label = LABELS.get(col, col)
    print(f"  {label:20s}  obs={n_obs:>5d}  missing={n_missing:>4d}")

# ── Diagnostic 2: Average annualised vol ──────────────────────────
ret = prices_df.pct_change().fillna(0.0)
vol_ann = tsmom.ex_ante_vol_ewma(ret)

print("\n" + "=" * 65)
print("  AVERAGE ANNUALISED VOL (EWMA)")
print("=" * 65)
for col in vol_ann.columns:
    avg_vol = vol_ann[col].dropna().mean()
    label = LABELS.get(col, col)
    print(f"  {label:20s}  {avg_vol:.1%}")

# ── Diagnostic 3: Return correlations ─────────────────────────────
corr = ret.rename(columns=LABELS).corr()
print("\n" + "=" * 65)
print("  DAILY RETURN CORRELATION MATRIX")
print("=" * 65)
print(corr.round(2).to_string())

# ── Diagnostic 4: Positions only change monthly ──────────────────
pos_changes = positions.diff().abs()
daily_change_count = (pos_changes > 0).sum(axis=1)
print(f"\n  Position changes occur on {(daily_change_count > 0).sum()} of "
      f"{len(daily_change_count)} trading days "
      f"({(daily_change_count > 0).mean():.1%})")

## Visualisations

1. Portfolio equity curve (net of costs)
2. Per-instrument equity curves
3. Monthly signal heatmap (+1 / −1 regime)
4. Daily turnover (should show monthly spikes)

In [ ]:
# --- 1. Portfolio Equity Curve ---
port = bt["portfolio"]
fig_port = go.Figure()
fig_port.add_trace(go.Scatter(
    x=port.index, y=port["equity"],
    name="Portfolio (net)",
    line=dict(color="#2c3e50", width=2),
    fill="tozeroy",
    fillcolor="rgba(44, 62, 80, 0.1)",
))
fig_port.update_layout(
    title="Canonical TSMOM — Portfolio Equity (IS: 2015-2022, $1M start)",
    template="plotly_white",
    hovermode="x unified",
    height=450,
    yaxis_title="Equity ($)",
    yaxis_tickformat="$,.0f",
)
fig_port.show()

# --- 2. Per-Instrument Equity Curves ---
equity_df = pd.DataFrame({
    LABELS.get(k, k): v["equity"]
    for k, v in bt["per_instrument"].items()
})
equity_norm = equity_df / equity_df.iloc[0]

fig_inst = px.line(
    equity_norm,
    title="Canonical TSMOM — Per-Instrument Equity (normalised to $1)",
    labels={"value": "Growth of $1", "variable": "Instrument", "date": ""},
)
fig_inst.update_layout(
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.15),
    height=500,
    yaxis_tickformat="$.2f",
)
fig_inst.show()

# --- 3. Monthly Signal Heatmap ---
sig_monthly = signals.resample("M").last().rename(columns=LABELS)

fig_heat = px.imshow(
    sig_monthly.T,
    color_continuous_scale=[[0, "#e74c3c"], [0.5, "#ecf0f1"], [1, "#2ecc71"]],
    zmin=-1, zmax=1,
    title="Monthly Signal Regime (+1 Long / -1 Short)",
    labels={"x": "", "y": "Instrument", "color": "Signal"},
    aspect="auto",
)
fig_heat.update_layout(
    template="plotly_white",
    height=400,
    xaxis=dict(dtick="M3", tickformat="%Y-%m"),
)
fig_heat.show()

# --- 4. Daily Turnover (should show monthly spikes) ---
total_turnover = positions.diff().abs().sum(axis=1)

fig_turn = go.Figure()
fig_turn.add_trace(go.Bar(
    x=total_turnover.index,
    y=total_turnover.values,
    marker_color="#3498db",
    opacity=0.7,
    name="Turnover",
))
fig_turn.update_layout(
    title="Daily Position Turnover (absolute, all instruments)",
    template="plotly_white",
    height=350,
    yaxis_title="Turnover (sum of |Δposition|)",
    hovermode="x unified",
)
fig_turn.show()

## Performance Summary

Compare canonical TSMOM portfolio metrics against memory file targets (§6.3).
Per-strategy Sharpe target is **> 0.5**.

In [ ]:
pm = bt["metrics_portfolio"]

comparison = pd.DataFrame({
    "Metric": [
        "Sharpe Ratio",
        "Annualised Vol",
        "Max Drawdown",
        "Calmar Ratio",
        "Hit Rate",
        "Ann. Turnover",
    ],
    "Target (§6.3)": [
        f"> {targets['sharpe_per_strategy']:.1f}",
        f"{targets['vol_range_annual'][0]:.0%} - {targets['vol_range_annual'][1]:.0%}",
        f"< {targets['max_drawdown_pct']:.0f}%",
        f"> {targets['calmar_ratio']:.1f}",
        f"> {targets['hit_rate_daily']:.0%}",
        f"< {targets['max_turnover_annual']:.0f}x",
    ],
    "TSMOM Portfolio": [
        f"{pm['Sharpe']:.2f}",
        f"{pm['Ann. Vol']:.1%}",
        f"{pm['Max DD']:.1%}",
        f"{pm['Calmar']:.2f}",
        f"{pm['Hit Rate']:.1%}",
        f"{pm['Ann. Turnover']:.1f}x",
    ],
}).set_index("Metric")

print("=" * 65)
print("  CANONICAL TSMOM vs MEMORY FILE TARGETS  (IS: 2015-2022)")
print("=" * 65)
comparison

## Export

Save signals, positions, equity, and summary to `outputs/`.

In [ ]:
output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

datestamp = datetime.now().strftime("%Y%m%d")

# Monthly signals
sig_path = output_dir / f"tsmom_signals_monthly_is_{datestamp}.csv"
signals.rename(columns=LABELS).to_csv(sig_path)

# Daily positions
pos_path = output_dir / f"tsmom_positions_daily_is_{datestamp}.csv"
positions.rename(columns=LABELS).to_csv(pos_path)

# Equity curves (portfolio + per-instrument)
eq_port = bt["portfolio"]["equity"]
eq_inst = pd.DataFrame({
    LABELS.get(k, k): v["equity"]
    for k, v in bt["per_instrument"].items()
})
eq_all = pd.concat([eq_port.rename("PORTFOLIO"), eq_inst], axis=1)
eq_path = output_dir / f"tsmom_equity_is_{datestamp}.csv"
eq_all.to_csv(eq_path)

# Summary HTML
html_path = output_dir / f"tsmom_summary_{datestamp}.html"
html_content = (
    "<h2>Canonical TSMOM — IS Performance (2015-2022)</h2>\n"
    "<p>Signal: sign(12M return), monthly rebalance, EWMA vol scaling</p>\n"
    + "<h3>Per-instrument + Portfolio Metrics</h3>\n"
    + fmt_df.to_html()
    + "<br><h3>vs Memory File Targets (§6.3)</h3>\n"
    + comparison.to_html()
)
with open(html_path, "w") as f:
    f.write(html_content)

print(f"Exported to {output_dir.resolve()}/")
print(f"  {sig_path.name:45s}  ({signals.shape[0]} rows x {signals.shape[1]} cols)")
print(f"  {pos_path.name:45s}  ({positions.shape[0]} rows)")
print(f"  {eq_path.name:45s}  ({eq_all.shape[0]} rows x {eq_all.shape[1]} cols)")
print(f"  {html_path.name}")
print(f"\nNotebook complete: {datetime.now():%Y-%m-%d %H:%M}")